In [2]:
import pandas as pd
%pwd

'/home/kartik-agarwal/Documents/DialogGPT/ml/research/PytorchDemo'

In [30]:
import os
dataset_dir = os.path.join(os.getcwd(), '../artifacts/data_ingestion/')
total_data = 0
for file in os.listdir(dataset_dir):
    if file!= 'archive.zip':
        total_data +=len(pd.read_table(os.path.join(dataset_dir, file), encoding='latin1', on_bad_lines='skip'))
total_data


36790

In [85]:
import os
dataset_dir = os.path.join(os.getcwd(), '../artifacts/data_ingestion/')
data_frames = []
for file in os.listdir(dataset_dir):
    if file!= 'archive.zip':
        data_frames.append(pd.read_table(os.path.join(dataset_dir, file), header=None, encoding='latin1', on_bad_lines='skip'))
df = pd.concat(data_frames, ignore_index=True)

df


,0
0,"Okay, hold on, don't shoot."
1,- You see where you're going? - Mm-hmm.
2,Okay.
3,"Now, let's worry about how you get there."
4,Gotta move your foot here.
...,...
36808,- Mine. - Power...
36809,has a purpose.
36810,Why are you doing this?
36811,"Because I see, at long last, what's wrong with..."


# Steps
- Data cleaning (regex)
- Create a Vocabulary
- Add <UNK>,<ST>,<ED> keywords in vocab
- Convert Dialogues to numerical embeddings
- Split each line for next word prediction
- Add padding
- do a test train split
- save the tensors for training on collab

In [86]:
import nltk

# Tokenization
nltk.download('punkt')
nltk.download('punkt_tab')

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from nltk.tokenize import word_tokenize

[nltk_data] Downloading package punkt to /home/kartik-
[nltk_data]     agarwal/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/kartik-
[nltk_data]     agarwal/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [87]:
import re
data_string = "".join(df.values.astype(str).flatten())
re.sub(r'\.', ' ',data_string)
data_string

'Okay, hold on, don\'t shoot.- You see where you\'re going? - Mm-hmm.Okay.Now, let\'s worry about how you get there.Gotta move your foot here.Point your toe this way.Your hips here.Okay?- Can you see? - Yeah.- Are you sure? - Mm-hmm.How about now? Can you see now?- No. - How about now?All right.Ready? Three fingers.Nice!Nice throw, kiddo.Here you go.Hey, you guys want mayo? Or mustard?Or both?Who puts mayo on a hot dog?Probably your brothers.Two mustard, please! Thanks, Mama.Got it!Nate, mayo or mustard?How about ketchup?Or ketchup. I got ketchup, too.Mind your elbow.Good job, hawk-eye.Go get your arrow.Hey, guys!Enough practice. Soup\'s on!All right.We\'re coming. We\'re hungry.Lila, let\'s go.Lila?Honey?Hey, babe?Babe?Babe?Boys!Boys!Laura!You don\'t need to do that.Because you\'re just holding the position.Oh, yeah.That was close.That\'s a goal. We are now one apiece.I would like to try again.We\'re tied up. Feel the tension?It\'s fun.That was terrible. Now you have a chance to win.A

In [88]:
def clean_text(text):
    if not isinstance(text, str):
        text = str(text)

    # 1. Keep letters, numbers, spaces, and core punctuation (. , ! ? ')
    # [^a-zA-Z0-9\s.,!?'] means "remove everything EXCEPT these"
    text = re.sub(r"[^a-zA-Z0-9\s.,!?']", " ", text)

    # 2. Fix the apostrophe issue (keep it between letters)
    # This removes ' if it's just hanging at the start/end of a word
    text = re.sub(r"(?<!\w)'|'(?!\w)", " ", text)

    # 3. Standardize whitespace
    text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()

    return text


df[0] = df[0].apply(clean_text)
df.head()

,0
0,"okay, hold on, don't shoot."
1,you see where you're going? mm hmm.
2,okay.
3,"now, let's worry about how you get there."
4,gotta move your foot here.


In [89]:
data_string = clean_text(data_string)
data_string

"okay, hold on, don't shoot. you see where you're going? mm hmm.okay.now, let's worry about how you get there.gotta move your foot here.point your toe this way.your hips here.okay? can you see? yeah. are you sure? mm hmm.how about now? can you see now? no. how about now?all right.ready? three fingers.nice!nice throw, kiddo.here you go.hey, you guys want mayo? or mustard?or both?who puts mayo on a hot dog?probably your brothers.two mustard, please! thanks, mama.got it!nate, mayo or mustard?how about ketchup?or ketchup. i got ketchup, too.mind your elbow.good job, hawk eye.go get your arrow.hey, guys!enough practice. soup's on!all right.we're coming. we're hungry.lila, let's go.lila?honey?hey, babe?babe?babe?boys!boys!laura!you don't need to do that.because you're just holding the position.oh, yeah.that was close.that's a goal. we are now one apiece.i would like to try again.we're tied up. feel the tension?it's fun.that was terrible. now you have a chance to win.and you've won.congratula

In [109]:

# build vocab
vocab = {'<unk>':0, 'start_token':1, 'end_token':2}
tokens = word_tokenize(data_string.lower())
for token in Counter(tokens).keys():
  if token not in vocab:
    vocab[token] = len(vocab)

vocab

{'<unk>': 0,
 'start_token': 1,
 'end_token': 2,
 'okay': 3,
 ',': 4,
 'hold': 5,
 'on': 6,
 'do': 7,
 "n't": 8,
 'shoot': 9,
 '.': 10,
 'you': 11,
 'see': 12,
 'where': 13,
 "'re": 14,
 'going': 15,
 '?': 16,
 'mm': 17,
 'hmm.okay.now': 18,
 'let': 19,
 "'s": 20,
 'worry': 21,
 'about': 22,
 'how': 23,
 'get': 24,
 'there.': 25,
 'got': 26,
 'ta': 27,
 'move': 28,
 'your': 29,
 'foot': 30,
 'here.point': 31,
 'toe': 32,
 'this': 33,
 'way.your': 34,
 'hips': 35,
 'here.okay': 36,
 'can': 37,
 'yeah': 38,
 'are': 39,
 'sure': 40,
 'hmm.how': 41,
 'now': 42,
 'no': 43,
 'all': 44,
 'right.ready': 45,
 'three': 46,
 'fingers.nice': 47,
 '!': 48,
 'nice': 49,
 'throw': 50,
 'kiddo.here': 51,
 'go.hey': 52,
 'guys': 53,
 'want': 54,
 'mayo': 55,
 'or': 56,
 'mustard': 57,
 'both': 58,
 'who': 59,
 'puts': 60,
 'a': 61,
 'hot': 62,
 'dog': 63,
 'probably': 64,
 'brothers.two': 65,
 'please': 66,
 'thanks': 67,
 'mama.got': 68,
 'it': 69,
 'nate': 70,
 'ketchup': 71,
 'i': 72,
 'too.mind': 7

In [110]:
len(vocab)

27153

In [111]:
data_frames = df.values.flatten().astype(str).tolist()

data_frames

["okay, hold on, don't shoot.",
 "you see where you're going? mm hmm.",
 'okay.',
 "now, let's worry about how you get there.",
 'gotta move your foot here.',
 'point your toe this way.',
 'your hips here.',
 'okay?',
 'can you see? yeah.',
 'are you sure? mm hmm.',
 'how about now? can you see now?',
 'no. how about now?',
 'all right.',
 'ready? three fingers.',
 'nice!',
 'nice throw, kiddo.',
 'here you go.',
 'hey, you guys want mayo? or mustard?',
 'or both?',
 'who puts mayo on a hot dog?',
 'probably your brothers.',
 'two mustard, please! thanks, mama.',
 'got it!',
 'nate, mayo or mustard?',
 'how about ketchup?',
 'or ketchup. i got ketchup, too.',
 'mind your elbow.',
 'good job, hawk eye.',
 'go get your arrow.',
 'hey, guys!',
 "enough practice. soup's on!",
 'all right.',
 "we're coming. we're hungry.",
 "lila, let's go.",
 'lila?',
 'honey?',
 'hey, babe?',
 'babe?',
 'babe?',
 'boys!',
 'boys!',
 'laura!',
 "you don't need to do that.",
 "because you're just holding th

In [112]:
def text_to_indices(sentence, vocab):

  numerical_sentence = []

  for token in sentence:
    if token in vocab:
      numerical_sentence.append(vocab[token])
    else:
      numerical_sentence.append(vocab['<unk>'])

  return numerical_sentence


In [113]:
input_numerical_sentences = []

for sentence in data_frames:
    sentence = f"start_token {sentence} end_token"
    # print(sentence)
    input_numerical_sentences.append(text_to_indices(word_tokenize(sentence.lower()), vocab))

In [114]:
input_numerical_sentences

[[1, 3, 4, 5, 6, 4, 7, 8, 9, 10, 2],
 [1, 11, 12, 13, 11, 14, 15, 16, 17, 2659, 10, 2],
 [1, 3, 10, 2],
 [1, 42, 4, 19, 20, 21, 22, 23, 11, 24, 303, 10, 2],
 [1, 26, 27, 28, 29, 30, 395, 10, 2],
 [1, 393, 29, 32, 33, 190, 10, 2],
 [1, 29, 35, 395, 10, 2],
 [1, 3, 16, 2],
 [1, 37, 11, 12, 16, 38, 10, 2],
 [1, 39, 11, 40, 16, 17, 2659, 10, 2],
 [1, 23, 22, 42, 16, 37, 11, 12, 42, 16, 2],
 [1, 43, 10, 23, 22, 42, 16, 2],
 [1, 44, 385, 10, 2],
 [1, 1406, 16, 46, 468, 10, 2],
 [1, 49, 48, 2],
 [1, 49, 50, 4, 0, 10, 2],
 [1, 395, 11, 244, 10, 2],
 [1, 88, 4, 11, 53, 54, 55, 16, 56, 57, 16, 2],
 [1, 56, 58, 16, 2],
 [1, 59, 60, 55, 6, 61, 62, 63, 16, 2],
 [1, 64, 29, 9656, 10, 2],
 [1, 480, 57, 4, 66, 48, 67, 4, 19718, 10, 2],
 [1, 26, 69, 48, 2],
 [1, 70, 4, 55, 56, 57, 16, 2],
 [1, 23, 22, 71, 16, 2],
 [1, 56, 71, 10, 72, 26, 71, 4, 358, 10, 2],
 [1, 511, 29, 0, 10, 2],
 [1, 268, 75, 4, 76, 1596, 10, 2],
 [1, 244, 24, 29, 7967, 10, 2],
 [1, 88, 4, 53, 48, 2],
 [1, 79, 80, 10, 81, 20, 6, 48,

In [115]:
len(input_numerical_sentences)

36813

In [116]:
training_sequence = []
for sentence in input_numerical_sentences:
  for i in range(1, len(sentence)):
    training_sequence.append(sentence[:i+1])

In [118]:
len(training_sequence)

314695

In [119]:
training_sequence[:5]

[[1, 3], [1, 3, 4], [1, 3, 4, 5], [1, 3, 4, 5, 6], [1, 3, 4, 5, 6, 4]]

In [120]:
len_list = []

for sequence in training_sequence:
  len_list.append(len(sequence))

max(len_list)

36

In [121]:
padded_training_sequence = []
for sequence in training_sequence:

  padded_training_sequence.append([0]*(max(len_list) - len(sequence)) + sequence)

In [124]:

padded_training_sequence = torch.as_tensor(padded_training_sequence, dtype=torch.long)

X = padded_training_sequence[:, :-1]
y = padded_training_sequence[:,-1]
torch.save(X, '../artifacts/data_processing/X.pt')
torch.save(y, '../artifacts/data_processing/y.pt')

In [126]:
print(padded_training_sequence[:5])

tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 3],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 3, 4],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 1, 3, 4, 5],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 1, 3, 4, 5, 6],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 1, 3, 4, 5, 6, 4]])
